# Assignment 6 — Real-Data Weather and Climate Trend Analysis
Authentic NASA POWER gridded daily estimates for the 25-field South Carolina cluster; no network access is used here.

In [ ]:
from pathlib import Path
import hashlib,json,pandas as pd
from IPython.display import display,Image
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/assignment-06').exists())
A=ROOT/'data/assignment-06'; S=A/'source'; O=A/'output'

## Source provenance and checksum validation

In [ ]:
m=json.loads((S/'source_manifest.json').read_text()); raw=S/'nasa_power_daily_raw.json'
assert m['status']=='success' and m['no_synthetic_fallback'] and hashlib.sha256(raw.read_bytes()).hexdigest()==m['raw_response']['sha256']
display(pd.DataFrame([m['request']]))

## Field and weather-point context
The acreage-weighted centroid is one representative point. NASA POWER is gridded and cannot resolve field-to-field meteorological differences or replace an on-site station.

In [ ]:
loc=json.loads((S/'field_location_summary.json').read_text()); dist=pd.read_csv(S/'field_to_weather_point_distances.csv',dtype={'field_id':str}); assert len(dist)==25 and dist.field_id.nunique()==25; display(loc,dist.head())

## Daily structure and data quality
Fill values are missing, never interpolated. Hot (≥35 °C maximum), frost (≤0 °C minimum), dry (<1 mm), and heavy-rain (≥25 mm) flags are descriptive—not universal crop-injury thresholds.

In [ ]:
daily=pd.read_csv(O/'tables/weather_daily_1991_2025.csv',parse_dates=['date']); quality=pd.read_csv(O/'tables/weather_data_quality.csv'); assert len(daily)==12784; display(daily.head(),quality)

## Aggregation rules
Temperature and humidity/radiation are means; precipitation is accumulated. Months need ≥90% valid days. Dry spells reset at annual and April–October boundaries.

In [ ]:
monthly=pd.read_csv(O/'tables/weather_monthly_1991_2025.csv'); annual=pd.read_csv(O/'tables/weather_annual_1991_2025.csv'); warm=pd.read_csv(O/'tables/weather_warm_season_1991_2025.csv'); assert len(monthly)==420 and len(annual)==35; display(monthly.head(),annual.tail())

## Seasonal climatology — baseline 1991–2020

In [ ]:
normals=pd.read_csv(O/'tables/climate_normals_1991_2020.csv'); display(normals); display(Image(filename=str(O/'dashboard_assets/dashboard_seasonal_climate.png')))

## Trends and precipitation anomalies

In [ ]:
trends=pd.read_csv(O/'tables/climate_trend_statistics.csv'); anomalies=pd.read_csv(O/'tables/climate_anomalies_1991_2025.csv'); recent=pd.read_csv(O/'tables/recent_anomalies_2021_2025.csv'); display(trends,recent); display(Image(filename=str(O/'dashboard_assets/dashboard_climate_trends_anomalies.png')))

## Warm-season weather-risk metrics
April–October is a consistent agricultural analytical window, not every crop’s exact phenology. Metrics support planning context but do not establish crop damage or causation.

In [ ]:
display(warm[['year','warm_season_hot_day_count','warm_season_longest_dry_spell_days','warm_season_precipitation_mm']].tail(10))

## Independent internal validation and interpretation cautions
Trends are local and descriptive; statistical significance is not agronomic importance or global attribution. Gridded NASA estimates are not station measurements.

In [ ]:
assert daily.date.is_unique and daily.date.is_monotonic_increasing
assert daily.date.min()==pd.Timestamp('1991-01-01') and daily.date.max()==pd.Timestamp('2025-12-31')
assert (daily.diurnal_temperature_range_C-(daily.t2m_max_C-daily.t2m_min_C)).abs().max()<1e-9
assert set(recent.year)==set(range(2021,2026))
print('All notebook internal assertions passed.')